In [2]:
import os
import re
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer,set_seed
from peft import LoraConfig,PeftModel
from datetime import datetime
from datasets import load_dataset,Dataset,DatasetDict
import matplotlib.pyplot as plt


c:\Users\KRISH\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
BASE_MODEL="meta-llama/Meta-Llama-3.1-8b"
PROJECT_NAME="pricer"
HF_USER="ed-donner"

RUN_NAME="2024-09-13_13.04.39"
PROJECT_RUN_NAME=f"{PROJECT_NAME}_{RUN_NAME}"
REVISION="e8d637df551603dc86cd7a1598a8f44af4d7ae36"
FINETUNED_MODEL=f"ed-donner/{PROJECT_RUN_NAME}"

DATASET_NAME=f"{HF_USER}/pricer-data"

QUANT_4_BIT=True

GREEN= "\033[92m"
YELLOW="\033[93m"
RED="\033[91m"
RESET="\033[0m"
COLOR_MAP={"red":RED,"green":GREEN,"yellow":YELLOW}





In [3]:
hf_token=os.getenv("HF_TOKEN")


In [4]:
dataset=load_dataset(DATASET_NAME)
train=dataset["train"]
test=dataset["test"]


In [5]:
if QUANT_4_BIT:
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
else:
    bnb_config=BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16
    )

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,trust_remote_code=True)
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="right"

base_model=AutoModelForCausalLM.from_pretrained(
BASE_MODEL,quantization_config=quantization_config,device_map="auto"
)
base_model.generation_config.pad_token_id=tokenizer.pad_token_id

if REVISION:
    fine_tuned_model=PeftModel.from_pretrained( # PARAMETER EFFICIENT FINE-TUNING
        base_model,FINETUNED_MODEL,revision=REVISION
    )
else:
    fine_tuned_model=PeftModel.from_pretrained(
        base_model,FINETUNED_MODEL
    )

print(f"Memory footprint of the fine-tuned model: {fine_tuned_model.get_memory_footprint() /1e6:.1f} MB")

In [ ]:
fine_tuned_model

## EVALUATION

In [8]:
def extract_prices(s):
    if "Price is $" in s:
        contents=s.split("Price is $")[1]
        contents=contents.replace(',','')
        match=re.search(r"[-+]?\d*\.\d+|\d+",contents)
        return float(match.group()) if match else None
    return 0

In [ ]:
extract_prices("Price is $a fake price of $1,234.56 and some text")

In [ ]:
# prediction function 
def model_predict(prompt):
    set_seed(42)
    inputs=tokenizer.encode(prompt,return_tensors="pt").to("cuda")
    attention_mask=torch.ones_like(inputs).to("cuda")
    outputs=fine_tuned_model.generate(inputs,attention_mask=attention_mask,max_new_tokens=3,num_return_sequences=1)
    response=tokenizer.decode(outputs[0])
    return extract_prices(response)


In [10]:
## improved model prediction

top_K=3
def improved_model_predict(prompt):
    set_seed(42)
    inputs=tokenizer.encode(prompt,return_tensors="pt").to("cuda")
    attention_mask=torch.ones_like(inputs,device="cuda")

    with torch.no_grad():
        outputs=fine_tuned_model(inputs,attention_mask=attention_mask)
        next_token_logits=outputs.logits[:,-1,:].to("cpu")

    next_token_probs=torch.softmax(next_token_logits,dim=-1)
    top_prob,top_token_id= next_token_probs.topk(top_K)
    prices,weights=[],[]

    for i in range(top_K):
        predicted_token=tokenizer.decode(top_token_id[0][i])
        probability=top_prob[0][i]
        try:
            result=float(predicted_token)
        except ValueError as e:
            result=0.0
        if result>0:
            prices.append(result)
            weights.append(probability)
    if not prices:
        return 0.0,0.0
    
    total=sum(weights)
    weighted_prices=[price*weight/total  for price,weight in zip(prices,weights)]

    return sum(weighted_prices)



In [ ]:

class Tester:
    def __init__(self, predictor, data, title="Model Evaluation", size=250):
        self.predictor = predictor
        # Prevent index errors if dataset is smaller than requested size
        self.data =  data
        self.size = min(size, len(data))
        self.title = title
        self.guesses = []
        self.truths = []
        self.errors = []
        self.sles = []
        self.colors = []

    def color_for(self, error, truth):
        # Prevent division by zero if truth is 0
        if truth == 0:
            return "green" if error == 0 else "red"
            
        pct_error = error / truth
        
        # Switched to 'and' logic so small cheap items aren't falsely marked green
        if error < 40 and pct_error < 0.2:
            return "green"
        elif error < 80 and pct_error < 0.5:
            return "orange"
        else:
            return "red"
        
    def run_datapoint(self, i):
        datapoint = self.data[i]
        
        # Fallback to "prompt" if "text" key doesn't exist
        text_content = datapoint.get("text", datapoint.get("prompt", ""))
        
        guess = self.predictor(datapoint["text"])
        truth = datapoint["price"]
        
        error = abs(guess - truth)
        
        # Math safety: handle potential log(0) issues by adding 1
        log_error = math.log(truth + 1) - math.log(guess + 1)
        sle = log_error ** 2
        
        color = self.color_for(error, truth)
        
        # Safely parse title snippet
        split_text = text_content.split("\n\n")
        raw_title = split_text[1] if len(split_text) > 1 else text_content
        title_snippet = raw_title[:20] + "..."
        
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)
        
        print(f"{COLOR_MAP[color]}{title_snippet:20} | Guess: {guess:8.2f} | Truth: {truth:8.2f} | Error: {error:8.2f} | SLE: {sle:8.4f}{RESET}")
    
    def chart(self, title):
        if not self.guesses:
            print("No data to chart.")
            return
            
        plt.figure(figsize=(10, 6))
        max_value = max(max(self.guesses), max(self.truths)) if self.guesses else 10
        
        # Perfect prediction identity line
        plt.plot([0, max_value], [0, max_value], color='deepskyblue', lw=2, alpha=0.6, label='Perfect Matches')
        plt.scatter(self.truths, self.guesses, c=self.colors, alpha=0.6)
        
        plt.xlabel('Ground Truth')
        plt.ylabel('Predicted')
        plt.xlim(0, max_value)
        plt.ylim(0, max_value)
        plt.title(title)
        plt.grid(True, alpha=0.3)
        plt.show()
    
    def report(self):
        if not self.errors:
            print("No data processed to generate a report.")
            return
        average_error = sum(self.errors) / len(self.errors)
        rmsle = math.sqrt(sum(self.sles) / len(self.sles))
        hits = sum(1 for color in self.colors if color == "green")
        
        full_title = f"{self.title}\nAvg Error: {average_error:.2f}, RMSLE: {rmsle:.4f}, Hits: {hits}/{len(self.colors)}"
        self.chart(full_title)
    
    def run(self):
        for i in range(self.size):
            self.run_datapoint(i)
        self.report()
    
    @classmethod
    def test(cls, function, data, title="Model Evaluation", size=250):
        cls(function, data, title=title, size=size).run()

In [ ]:
Tester.test(improved_model_predict, test, title="Improved Model Evaluation")